# FNO vs MRBNO Lift v3: Synthetic Functions Comparison

This notebook compares **FNO** (Fourier Neural Operator) and **MRBNO Lift v3** on challenging synthetic functions that test the models' ability to handle:
- **Discontinuous functions** (step function)
- **Periodic non-smooth functions** (sawtooth)
- **High-frequency patterns** (chessboard/periodic step)

These functions are particularly challenging for FNO due to their discontinuities and high-frequency content, while MRBNO's PCA-based approach may better capture these patterns.

**Models:**
- **FNO**: width=24, modes=(16, 16), layers=4
- **MRBNO Lift v3**: width=24, layers=4, auto PCA modes (epsilon=1e-5)

**Training settings:**
- Cosine annealing LR schedule with lr=2e-3
- Gradient clipping: max_grad_norm=3.0
- Early stopping: patience=30
- Batch size: 32


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import sys
import os
import time

# Add models directory to path
sys.path.append('models')

from FNO_model import FNO2d
from MRBNO_lift_v3 import MRBNOLiftV3
from MRBNO_simple import PCAReducerMultiChannel

print("Libraries imported successfully!")


## 1. Generate Synthetic Functions Dataset


In [ ]:
def generate_step_function(N=64, num_samples=1000, random_shift=True):
    """Generate step function: u(x,y) = 1 if x > threshold, else 0"""
    X = np.zeros((num_samples, 2, N, N))
    Y = np.zeros((num_samples, N, N))
    
    x = np.linspace(0, 1, N)
    y = np.linspace(0, 1, N)
    X_grid, Y_grid = np.meshgrid(x, y, indexing='ij')
    
    for i in range(num_samples):
        if random_shift:
            threshold = np.random.uniform(0.3, 0.7)
        else:
            threshold = 0.5
        
        # Input: constant function f(x,y) = 1, m(x,y) = threshold
        X[i, 0] = np.ones((N, N))  # f(x,y) = 1
        X[i, 1] = threshold * np.ones((N, N))  # m(x,y) = threshold
        
        # Output: step function
        Y[i] = (X_grid > threshold).astype(float)
    
    return X, Y

def generate_sawtooth_function(N=64, num_samples=1000, random_freq=True):
    """Generate sawtooth function: u(x,y) = sawtooth(x)"""
    X = np.zeros((num_samples, 2, N, N))
    Y = np.zeros((num_samples, N, N))
    
    x = np.linspace(0, 1, N)
    y = np.linspace(0, 1, N)
    X_grid, Y_grid = np.meshgrid(x, y, indexing='ij')
    
    for i in range(num_samples):
        if random_freq:
            freq = np.random.uniform(2, 8)
        else:
            freq = 4
        
        # Input: f(x,y) = freq, m(x,y) = 0
        X[i, 0] = freq * np.ones((N, N))
        X[i, 1] = np.zeros((N, N))
        
        # Output: sawtooth wave
        sawtooth = (X_grid * freq) % 1.0
        Y[i] = sawtooth
    
    return X, Y

def generate_chessboard_function(N=64, num_samples=1000, random_grid=True):
    """Generate chessboard pattern: periodic 0/1 pattern"""
    X = np.zeros((num_samples, 2, N, N))
    Y = np.zeros((num_samples, N, N))
    
    x = np.linspace(0, 1, N)
    y = np.linspace(0, 1, N)
    X_grid, Y_grid = np.meshgrid(x, y, indexing='ij')
    
    for i in range(num_samples):
        if random_grid:
            grid_size = np.random.choice([4, 6, 8, 10])
        else:
            grid_size = 8
        
        # Input: f(x,y) = grid_size, m(x,y) = 0
        X[i, 0] = grid_size * np.ones((N, N))
        X[i, 1] = np.zeros((N, N))
        
        # Output: chessboard pattern
        chessboard = ((X_grid * grid_size).astype(int) + (Y_grid * grid_size).astype(int)) % 2
        Y[i] = chessboard.astype(float)
    
    return X, Y

# Generate datasets
print("Generating synthetic datasets...")
N = 64
num_samples_per_function = 1000

X_step, Y_step = generate_step_function(N=N, num_samples=num_samples_per_function)
X_sawtooth, Y_sawtooth = generate_sawtooth_function(N=N, num_samples=num_samples_per_function)
X_chessboard, Y_chessboard = generate_chessboard_function(N=N, num_samples=num_samples_per_function)

print(f"Step function: X={X_step.shape}, Y={Y_step.shape}")
print(f"Sawtooth function: X={X_sawtooth.shape}, Y={Y_sawtooth.shape}")
print(f"Chessboard function: X={X_chessboard.shape}, Y={Y_chessboard.shape}")

# Visualize samples
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

# Step function
axes[0, 0].imshow(X_step[0, 0], origin='lower', cmap='viridis')
axes[0, 0].set_title('Step Input: f(x,y)')
axes[0, 1].imshow(X_step[0, 1], origin='lower', cmap='viridis')
axes[0, 1].set_title('Step Input: m(x,y)')
axes[0, 2].imshow(Y_step[0], origin='lower', cmap='coolwarm')
axes[0, 2].set_title('Step Output: u(x,y)')

# Sawtooth function
axes[1, 0].imshow(X_sawtooth[0, 0], origin='lower', cmap='viridis')
axes[1, 0].set_title('Sawtooth Input: f(x,y)')
axes[1, 1].imshow(X_sawtooth[0, 1], origin='lower', cmap='viridis')
axes[1, 1].set_title('Sawtooth Input: m(x,y)')
axes[1, 2].imshow(Y_sawtooth[0], origin='lower', cmap='coolwarm')
axes[1, 2].set_title('Sawtooth Output: u(x,y)')

# Chessboard function
axes[2, 0].imshow(X_chessboard[0, 0], origin='lower', cmap='viridis')
axes[2, 0].set_title('Chessboard Input: f(x,y)')
axes[2, 1].imshow(X_chessboard[0, 1], origin='lower', cmap='viridis')
axes[2, 1].set_title('Chessboard Input: m(x,y)')
axes[2, 2].imshow(Y_chessboard[0], origin='lower', cmap='coolwarm')
axes[2, 2].set_title('Chessboard Output: u(x,y)')

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()


## 2. Training Function


In [ ]:
def train_model(model, train_loader, val_loader, epochs=300, lr=2e-3, weight_decay=1e-4, 
                max_grad_norm=3.0, device='cpu', early_stopping_patience=30, model_name="model"):
    """Train model with cosine annealing LR schedule and early stopping."""
    model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
        betas=(0.9, 0.999),
        eps=1e-8
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=0.0)
    criterion = nn.MSELoss()
    
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    best_epoch = 0
    patience_counter = 0
    
    start_time = time.time()
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(train_loader.dataset)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                loss = criterion(pred, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(val_loader.dataset)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1
        
        scheduler.step()
        
        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"[{model_name}] Epoch {epoch+1:03d}/{epochs}: train={train_loss:.4e} val={val_loss:.4e} "
                  f"lr={scheduler.get_last_lr()[0]:.2e} (best: {best_val_loss:.4e} @ epoch {best_epoch+1})")
        
        if patience_counter >= early_stopping_patience:
            print(f"[{model_name}] Early stopping at epoch {epoch+1}")
            break
    
    training_time = time.time() - start_time
    return train_losses, val_losses, best_val_loss, training_time, best_epoch + 1

def compute_metrics(model, test_loader, device, y_std=1.0):
    """Compute test metrics: MSE, MAE, Relative L1, Relative L2"""
    model.eval()
    mse_loss = nn.MSELoss()
    mae_loss = nn.L1Loss()
    
    total_mse = 0.0
    total_mae = 0.0
    total_rel_l1 = 0.0
    total_rel_l2 = 0.0
    
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            
            # Denormalize
            pred_orig = pred * y_std
            yb_orig = yb * y_std
            
            # MSE and MAE
            mse = mse_loss(pred_orig, yb_orig).item()
            mae = mae_loss(pred_orig, yb_orig).item()
            
            # Relative errors
            pred_flat = pred_orig.view(pred_orig.shape[0], -1)
            yb_flat = yb_orig.view(yb_orig.shape[0], -1)
            
            rel_l1 = torch.abs(pred_flat - yb_flat).sum(dim=1) / (torch.abs(yb_flat).sum(dim=1) + 1e-8)
            rel_l2 = torch.norm(pred_flat - yb_flat, dim=1) / (torch.norm(yb_flat, dim=1) + 1e-8)
            
            total_mse += mse * xb.size(0)
            total_mae += mae * xb.size(0)
            total_rel_l1 += rel_l1.sum().item()
            total_rel_l2 += rel_l2.sum().item()
    
    n_samples = len(test_loader.dataset)
    return {
        'mse': total_mse / n_samples,
        'mae': total_mae / n_samples,
        'rel_l1': total_rel_l1 / n_samples,
        'rel_l2': total_rel_l2 / n_samples
    }


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else 
                      "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Results storage
results = {}

# Process each function type
function_types = {
    'step': (X_step, Y_step, 'Step Function'),
    'sawtooth': (X_sawtooth, Y_sawtooth, 'Sawtooth Function'),
    'chessboard': (X_chessboard, Y_chessboard, 'Chessboard Function')
}

for func_name, (X, Y, func_title) in function_types.items():
    print(f"\n{'='*80}")
    print(f"PROCESSING: {func_title}")
    print(f"{'='*80}")
    
    # Split data: 70% train, 20% val, 10% test
    n_total = len(X)
    n_train = int(0.7 * n_total)
    n_val = int(0.2 * n_total)
    
    X_train, Y_train = X[:n_train], Y[:n_train]
    X_val, Y_val = X[n_train:n_train+n_val], Y[n_train:n_train+n_val]
    X_test, Y_test = X[n_train+n_val:], Y[n_train+n_val:]
    
    # Normalize
    x_mean, x_std = X_train.mean(), X_train.std()
    y_mean, y_std = Y_train.mean(), Y_train.std()
    
    X_train_norm = (X_train - x_mean) / (x_std + 1e-8)
    Y_train_norm = (Y_train - y_mean) / (y_std + 1e-8)
    X_val_norm = (X_val - x_mean) / (x_std + 1e-8)
    Y_val_norm = (Y_val - y_mean) / (y_std + 1e-8)
    X_test_norm = (X_test - x_mean) / (x_std + 1e-8)
    Y_test_norm = (Y_test - y_mean) / (y_std + 1e-8)
    
    # Convert to tensors
    X_train_t = torch.from_numpy(X_train_norm).float()
    Y_train_t = torch.from_numpy(Y_train_norm).float().unsqueeze(1)
    X_val_t = torch.from_numpy(X_val_norm).float()
    Y_val_t = torch.from_numpy(Y_val_norm).float().unsqueeze(1)
    X_test_t = torch.from_numpy(X_test_norm).float()
    Y_test_t = torch.from_numpy(Y_test_norm).float().unsqueeze(1)
    
    # Data loaders
    batch_size = 32
    train_loader = DataLoader(TensorDataset(X_train_t, Y_train_t), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val_t, Y_val_t), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(TensorDataset(X_test_t, Y_test_t), batch_size=batch_size, shuffle=False)
    
    results[func_name] = {}
    
    # Train FNO
    print(f"\n--- Training FNO on {func_title} ---")
    fno_model = FNO2d(in_channels=2, out_channels=1, width=24, modes=(16, 16), n_layers=4)
    fno_params = sum(p.numel() for p in fno_model.parameters())
    print(f"FNO Parameters: {fno_params:,}")
    
    fno_train_losses, fno_val_losses, fno_best_val, fno_time, fno_best_epoch = train_model(
        fno_model, train_loader, val_loader, epochs=300, lr=2e-3, weight_decay=1e-4,
        max_grad_norm=3.0, device=device, early_stopping_patience=30, model_name="FNO"
    )
    
    fno_metrics = compute_metrics(fno_model, test_loader, device, y_std=y_std)
    print(f"FNO Test Metrics: MSE={fno_metrics['mse']:.4e}, MAE={fno_metrics['mae']:.4e}, "
          f"Rel L1={fno_metrics['rel_l1']:.4f}, Rel L2={fno_metrics['rel_l2']:.4f}")
    
    results[func_name]['fno'] = {
        'train_losses': fno_train_losses,
        'val_losses': fno_val_losses,
        'best_val_loss': fno_best_val,
        'test_metrics': fno_metrics,
        'params': fno_params,
        'training_time': fno_time,
        'best_epoch': fno_best_epoch,
        'model': fno_model
    }
    
    # Setup PCA for MRBNO (use original unnormalized data for PCA fitting)
    print(f"\n--- Setting up PCA for MRBNO on {func_title} ---")
    rb = PCAReducerMultiChannel(
        bc_weight=1.0,
        boundary_threshold=0.1,
        use_boundary_weighting=False
    ).fit(X_train, Y_train)  # Use original data for PCA
    
    epsilon = 1e-5
    k_in, k_out = rb.k_from_cutoff(j_in=1-epsilon, j_out=1-epsilon)
    print(f"PCA components: k_in={k_in}, k_out={k_out}")
    
    # Train MRBNO Lift v3
    print(f"\n--- Training MRBNO Lift v3 on {func_title} ---")
    mrbno_model = MRBNOLiftV3(
        width=24,
        rb=rb,
        k_in=k_in,
        k_out=k_out,
        layers=4,
        dropout=0.0,
        init_scale=0.1,
        use_skip_connections=True,
        use_batch_norm=False
    )
    mrbno_params = sum(p.numel() for p in mrbno_model.parameters())
    print(f"MRBNO Lift v3 Parameters: {mrbno_params:,}")
    
    mrbno_train_losses, mrbno_val_losses, mrbno_best_val, mrbno_time, mrbno_best_epoch = train_model(
        mrbno_model, train_loader, val_loader, epochs=300, lr=2e-3, weight_decay=1e-4,
        max_grad_norm=3.0, device=device, early_stopping_patience=30, model_name="MRBNO"
    )
    
    mrbno_metrics = compute_metrics(mrbno_model, test_loader, device, y_std=y_std)
    print(f"MRBNO Test Metrics: MSE={mrbno_metrics['mse']:.4e}, MAE={mrbno_metrics['mae']:.4e}, "
          f"Rel L1={mrbno_metrics['rel_l1']:.4f}, Rel L2={mrbno_metrics['rel_l2']:.4f}")
    
    results[func_name]['mrbno'] = {
        'train_losses': mrbno_train_losses,
        'val_losses': mrbno_val_losses,
        'best_val_loss': mrbno_best_val,
        'test_metrics': mrbno_metrics,
        'params': mrbno_params,
        'training_time': mrbno_time,
        'best_epoch': mrbno_best_epoch,
        'k_in': k_in,
        'k_out': k_out,
        'model': mrbno_model
    }
    
    # Clear memory
    del fno_model, mrbno_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None


## 4. Results Summary and Visualization


In [ ]:
# Print summary table
print("\n" + "="*100)
print("RESULTS SUMMARY")
print("="*100)
print(f"{'Function':<15} {'Model':<15} {'MSE':<12} {'MAE':<12} {'Rel L1':<10} {'Rel L2':<10} {'Params':<12} {'Time (s)':<10}")
print("-"*100)

for func_name in ['step', 'sawtooth', 'chessboard']:
    func_title = function_types[func_name][2]
    for model_name in ['fno', 'mrbno']:
        r = results[func_name][model_name]
        m = r['test_metrics']
        print(f"{func_title:<15} {model_name.upper():<15} {m['mse']:<12.4e} {m['mae']:<12.4e} "
              f"{m['rel_l1']:<10.4f} {m['rel_l2']:<10.4f} {r['params']:<12,} {r['training_time']:<10.1f}")

# Plot learning curves
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
for idx, func_name in enumerate(['step', 'sawtooth', 'chessboard']):
    func_title = function_types[func_name][2]
    
    # Training curves
    axes[idx, 0].plot(results[func_name]['fno']['train_losses'], label='FNO Train', alpha=0.7)
    axes[idx, 0].plot(results[func_name]['fno']['val_losses'], label='FNO Val', linestyle='--', alpha=0.7)
    axes[idx, 0].plot(results[func_name]['mrbno']['train_losses'], label='MRBNO Train', alpha=0.7)
    axes[idx, 0].plot(results[func_name]['mrbno']['val_losses'], label='MRBNO Val', linestyle='--', alpha=0.7)
    axes[idx, 0].set_xlabel('Epoch')
    axes[idx, 0].set_ylabel('Loss (MSE)')
    axes[idx, 0].set_title(f'{func_title} - Learning Curves')
    axes[idx, 0].legend()
    axes[idx, 0].grid(True, alpha=0.3)
    axes[idx, 0].set_yscale('log')
    
    # Test metrics comparison
    fno_metrics = results[func_name]['fno']['test_metrics']
    mrbno_metrics = results[func_name]['mrbno']['test_metrics']
    
    metrics_names = ['MSE', 'MAE', 'Rel L1', 'Rel L2']
    fno_values = [fno_metrics['mse'], fno_metrics['mae'], fno_metrics['rel_l1'], fno_metrics['rel_l2']]
    mrbno_values = [mrbno_metrics['mse'], mrbno_metrics['mae'], mrbno_metrics['rel_l1'], mrbno_metrics['rel_l2']]
    
    x = np.arange(len(metrics_names))
    width = 0.35
    axes[idx, 1].bar(x - width/2, fno_values, width, label='FNO', alpha=0.7)
    axes[idx, 1].bar(x + width/2, mrbno_values, width, label='MRBNO Lift v3', alpha=0.7)
    axes[idx, 1].set_xlabel('Metric')
    axes[idx, 1].set_ylabel('Value')
    axes[idx, 1].set_title(f'{func_title} - Test Metrics')
    axes[idx, 1].set_xticks(x)
    axes[idx, 1].set_xticklabels(metrics_names)
    axes[idx, 1].legend()
    axes[idx, 1].grid(True, alpha=0.3, axis='y')
    axes[idx, 1].set_yscale('log')

plt.tight_layout()
plt.show()

# Plot output snapshots: MRBNO vs FNO vs Ground Truth
def plot_prediction_snapshots(results, function_types, X_test_data, Y_test_data, 
                              y_mean, y_std, num_samples=5, device='cpu'):
    """
    Plot prediction snapshots comparing MRBNO, FNO, and ground truth.
    
    Args:
        results: Dictionary containing model results for each function type
        function_types: Dictionary mapping function names to (X, Y, title) tuples
        X_test_data: Dictionary of test input data for each function type (normalized)
        Y_test_data: Dictionary of test output data for each function type (normalized)
        y_mean: Dictionary of y means for each function type
        y_std: Dictionary of y stds for each function type
        num_samples: Number of samples to visualize per function type
        device: Device to run models on
    """
    for func_name in ['step', 'sawtooth', 'chessboard']:
        func_title = function_types[func_name][2]
        
        # Get models
        fno_model = results[func_name]['fno']['model']
        mrbno_model = results[func_name]['mrbno']['model']
        
        # Get test data (already normalized)
        X_test_norm = X_test_data[func_name]
        Y_test_norm = Y_test_data[func_name]
        
        # Convert to tensors if needed
        if isinstance(X_test_norm, np.ndarray):
            X_test_t = torch.from_numpy(X_test_norm).float()
        else:
            X_test_t = X_test_norm
        
        if isinstance(Y_test_norm, np.ndarray):
            Y_test_t = torch.from_numpy(Y_test_norm).float()
        else:
            Y_test_t = Y_test_norm
        
        # Denormalization values
        y_mean_val = y_mean[func_name]
        y_std_val = y_std[func_name]
        
        # Select random samples
        n_test = len(X_test_t)
        sample_indices = np.random.choice(n_test, min(num_samples, n_test), replace=False)
        
        # Create figure
        fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
        if num_samples == 1:
            axes = axes.reshape(1, -1)
        
        fno_model.to(device)
        mrbno_model.to(device)
        
        with torch.no_grad():
            for row, idx in enumerate(sample_indices):
                # Get sample
                x_sample = X_test_t[idx:idx+1].to(device)  # (1, 2, N, N)
                y_true_norm = Y_test_t[idx]  # (N, N)
                
                # Get predictions
                fno_pred_norm = fno_model(x_sample).cpu().numpy()[0, 0]  # (N, N)
                mrbno_pred_norm = mrbno_model(x_sample).cpu().numpy()[0, 0]  # (N, N)
                
                # Denormalize
                y_true = y_true_norm.numpy() * y_std_val + y_mean_val
                fno_pred = fno_pred_norm * y_std_val + y_mean_val
                mrbno_pred = mrbno_pred_norm * y_std_val + y_mean_val
                
                # Compute errors
                fno_rel_l2 = np.linalg.norm(fno_pred - y_true) / (np.linalg.norm(y_true) + 1e-10)
                mrbno_rel_l2 = np.linalg.norm(mrbno_pred - y_true) / (np.linalg.norm(y_true) + 1e-10)
                
                # Ground truth
                im0 = axes[row, 0].imshow(y_true, origin='lower', cmap='coolwarm')
                axes[row, 0].set_title(f'Ground Truth (Sample {idx})')
                axes[row, 0].set_xticks([])
                axes[row, 0].set_yticks([])
                plt.colorbar(im0, ax=axes[row, 0], fraction=0.046)
                
                # FNO prediction
                im1 = axes[row, 1].imshow(fno_pred, origin='lower', cmap='coolwarm')
                axes[row, 1].set_title(f'FNO (Rel L2: {fno_rel_l2:.4f})')
                axes[row, 1].set_xticks([])
                axes[row, 1].set_yticks([])
                plt.colorbar(im1, ax=axes[row, 1], fraction=0.046)
                
                # MRBNO prediction
                im2 = axes[row, 2].imshow(mrbno_pred, origin='lower', cmap='coolwarm')
                axes[row, 2].set_title(f'MRBNO Lift v3 (Rel L2: {mrbno_rel_l2:.4f})')
                axes[row, 2].set_xticks([])
                axes[row, 2].set_yticks([])
                plt.colorbar(im2, ax=axes[row, 2], fraction=0.046)
                
                # Error comparison
                fno_error = np.abs(fno_pred - y_true)
                mrbno_error = np.abs(mrbno_pred - y_true)
                error_diff = mrbno_error - fno_error  # Positive = MRBNO worse, Negative = MRBNO better
                
                im3 = axes[row, 3].imshow(error_diff, origin='lower', cmap='RdBu_r', 
                                         vmin=-error_diff.max(), vmax=error_diff.max())
                axes[row, 3].set_title('Error Diff (MRBNO - FNO)\nBlue = MRBNO better, Red = FNO better')
                axes[row, 3].set_xticks([])
                axes[row, 3].set_yticks([])
                plt.colorbar(im3, ax=axes[row, 3], fraction=0.046)
        
        plt.suptitle(f'{func_title} - Prediction Snapshots', fontsize=16, y=0.995)
        plt.tight_layout()
        plt.show()

# Store test data and normalization stats for plotting
X_test_data = {}
Y_test_data = {}
y_mean_dict = {}
y_std_dict = {}

# Need to collect this during the training loop, so let's modify the loop or re-extract
# For now, let's extract from results if available, otherwise recreate
print("\n" + "="*80)
print("PLOTTING PREDICTION SNAPSHOTS")
print("="*80)

# Re-extract test data and normalization stats
for func_name, (X, Y, func_title) in function_types.items():
    n_total = len(X)
    n_train = int(0.7 * n_total)
    n_val = int(0.2 * n_total)
    
    X_train = X[:n_train]
    Y_train = Y[:n_train]
    X_test = X[n_train+n_val:]
    Y_test = Y[n_train+n_val:]
    
    # Normalize using train stats
    x_mean, x_std = X_train.mean(), X_train.std()
    y_mean_val, y_std_val = Y_train.mean(), Y_train.std()
    
    X_test_norm = (X_test - x_mean) / (x_std + 1e-8)
    Y_test_norm = (Y_test - y_mean_val) / (y_std_val + 1e-8)
    
    X_test_data[func_name] = X_test_norm
    Y_test_data[func_name] = Y_test_norm
    y_mean_dict[func_name] = y_mean_val
    y_std_dict[func_name] = y_std_val

# Plot snapshots for each function type
plot_prediction_snapshots(results, function_types, X_test_data, Y_test_data, 
                          y_mean_dict, y_std_dict, num_samples=3, device=device)
